# 从零实现 Apriori

本教程使用一个小型菜单订单数据集，手写 Apriori 的核心流程，帮助理解频繁项集和关联规则的计算过程。


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    display = print


RANDOM_STATE = 42


def find_project_root(start=None):
    """Find the project root by walking upward to the data directory.

    Args:
        start: Optional start path. Defaults to the current working directory.

    Returns:
        Project root path containing the `data` directory.

    Raises:
        FileNotFoundError: If no parent directory contains `data`.
    """
    current = Path.cwd() if start is None else Path(start).resolve()
    for path in [current, *current.parents]:
        if (path / "data").exists():
            return path
    raise FileNotFoundError("未找到包含 data/ 的项目根目录")


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"

plt.rcParams["font.sans-serif"] = ["SimHei", "Arial Unicode MS", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False


from itertools import combinations


## 1. 构建购物篮矩阵


In [ ]:
raw_orders = pd.read_excel(DATA_DIR / "menu_orders.xls", header=None)
transactions = raw_orders.apply(lambda row: sorted(row.dropna().astype(str)), axis=1).tolist()
items = sorted({item for transaction in transactions for item in transaction})

basket = pd.DataFrame(
    [{item: item in transaction for item in items} for transaction in transactions],
    dtype=bool,
)

print(f"订单数: {len(transactions)}")
print(f"商品数: {len(items)}")
display(basket.astype(int))


## 2. 实现 Apriori


In [ ]:
def itemset_support(basket, itemset):
    """Compute the support of an itemset in a basket matrix.

    Args:
        basket: Boolean transaction-item matrix.
        itemset: Items whose joint occurrence is measured.

    Returns:
        Fraction of transactions containing every item in the itemset.
    """
    columns = list(itemset)
    return basket[columns].all(axis=1).mean()


def generate_candidates(previous_itemsets, size):
    """Generate candidate itemsets from frequent itemsets in the previous layer.

    Args:
        previous_itemsets: Frequent itemsets from the previous Apriori layer.
        size: Target size of the generated candidate itemsets.

    Returns:
        Sorted list of candidate itemsets represented as frozensets.
    """
    candidates = set()
    previous = list(previous_itemsets)
    for left, right in combinations(previous, 2):
        candidate = left | right
        if len(candidate) == size:
            candidates.add(candidate)
    return sorted(candidates, key=lambda itemset: tuple(sorted(itemset)))


def apriori_frequent_itemsets(basket, min_support=0.5):
    """Mine frequent itemsets with the Apriori algorithm.

    Args:
        basket: Boolean transaction-item matrix.
        min_support: Minimum support threshold.

    Returns:
        A tuple containing the support dictionary and a display-ready report.
    """
    frequent = {}
    current = []

    for column in basket.columns:
        itemset = frozenset([column])
        support = itemset_support(basket, itemset)
        if support >= min_support:
            frequent[itemset] = support
            current.append(itemset)

    size = 2
    while current:
        candidates = generate_candidates(current, size)
        current = []
        for candidate in candidates:
            support = itemset_support(basket, candidate)
            if support >= min_support:
                frequent[candidate] = support
                current.append(candidate)
        size += 1

    result = pd.DataFrame(
        [
            {"itemset": ", ".join(sorted(itemset)), "support": support}
            for itemset, support in frequent.items()
        ]
    )
    return frequent, result.sort_values(["support", "itemset"], ascending=[False, True])


frequent_itemsets, frequent_report = apriori_frequent_itemsets(basket, min_support=0.5)
display(frequent_report)


## 3. 生成关联规则


In [ ]:
def generate_rules(frequent_itemsets, min_confidence=0.5):
    """Generate association rules from frequent itemsets.

    Args:
        frequent_itemsets: Mapping from itemsets to support values.
        min_confidence: Minimum confidence threshold.

    Returns:
        Data frame sorted by confidence, support, and lift.
    """
    records = []
    for itemset, support in frequent_itemsets.items():
        if len(itemset) < 2:
            continue
        for size in range(1, len(itemset)):
            for antecedent_items in combinations(itemset, size):
                antecedent = frozenset(antecedent_items)
                consequent = itemset - antecedent
                antecedent_support = frequent_itemsets[antecedent]
                consequent_support = frequent_itemsets[consequent]
                confidence = support / antecedent_support
                lift = confidence / consequent_support
                if confidence >= min_confidence:
                    records.append(
                        {
                            "antecedent": ", ".join(sorted(antecedent)),
                            "consequent": ", ".join(sorted(consequent)),
                            "support": support,
                            "confidence": confidence,
                            "lift": lift,
                        }
                    )

    return pd.DataFrame(records).sort_values(
        ["confidence", "support", "lift"], ascending=False
    )


rules = generate_rules(frequent_itemsets, min_confidence=0.5)
display(rules.round(4))
